# 🏥 Preferential Mixture of Experts — ICU 24h Mortality
**Base Learner:** PyTorch Neural Network (4 hidden layers, BatchNorm, GELU, Cosine LR)

**Models compared:**
- Neural Network Only
- Standard MoE
- Preferential MoE (log-barrier)
- Preferential MoE (KL-reg)

**Evaluation:** 5-seed × 5-fold cross-validation for robustness

## 📦 Cell 1 — Install & Imports

In [ ]:
# !pip install -q torch scikit-learn scipy pandas numpy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.impute import KNNImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from scipy.special import expit
from scipy.optimize import minimize
import itertools, copy, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')
print(f'   PyTorch version: {torch.__version__}')
print(f'   NumPy version:   {np.__version__}')
print(f'   Pandas version:  {pd.__version__}')

## 📁 Cell 2 — Upload & Inspect Data

In [ ]:
# from google.colab import files
# import io

# print('📂 Please upload your CSV file (e.g. icu24h_mortality_imputed12.csv)...')
# uploaded = files.upload()

# Read whichever file was uploaded
# filename = list(uploaded.keys())[0]
# df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

import pandas as pd
import os

# Define the path to your data folder (standard for research papers)
data_path = os.path.join('data', 'icu24h_mortality_imputed12.csv')

# Check if the file exists before attempting to load
if os.path.exists(data_path):
    df_raw = pd.read_csv(data_path)
    print(f"✅ Loaded MIMIC-III data successfully from: {data_path}")
    print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
else:
    print(f"❌ Error: Could not find '{data_path}'.")
    print("👉 Please ensure you have a folder named 'data' with the CSV file inside it.")

# Preview the first few rows
if 'df_raw' in locals():
    print(df_raw.head())

# Label detection
LABEL_COL = 'label_24h_icu_mortality'
if LABEL_COL not in df_raw.columns:
    LABEL_COL = 'label_in_hosp_mortality'
    print(f'   ⚠️  label_24h_icu_mortality not found — using {LABEL_COL}')
else:
    print(f'   Label column: {LABEL_COL}')

print(f'\n📊 Class distribution:')
vc = df_raw[LABEL_COL].value_counts()
print(f'   Negative (0): {vc.get(0, 0):,}  ({vc.get(0,0)/len(df_raw)*100:.1f}%)')
print(f'   Positive (1): {vc.get(1, 0):,}  ({vc.get(1,0)/len(df_raw)*100:.1f}%)')

print(f'\n🔍 Missing values: {df_raw.isnull().sum().sum():,} total')
miss = df_raw.isnull().mean()
miss_cols = miss[miss > 0].sort_values(ascending=False)
if len(miss_cols) > 0:
    print(f'   Columns with missing data ({len(miss_cols)} total, top 10):')
    print(miss_cols.head(10).to_string())
else:
    print('   No missing values found.')

print(f'\n📋 Column overview:')
print(f'   Numeric columns:  {df_raw.select_dtypes(include="number").shape[1]}')
print(f'   Object columns:   {df_raw.select_dtypes(include="object").shape[1]}')
print()
df_raw.head(3)

## ⚙️ Cell 3 — Configuration & Hyperparameters

In [ ]:
# ── MoE hyperparameters ──────────────────────────────────────────
GAMMA      = 0.0001
BETA_KL    = 1.0
MU_COV     = 0.1
EPS        = 0.01
T          = 8.0
N_RESTARTS = 2
TEST_SIZE  = 0.20

# ── Neural network hyperparameters ───────────────────────────────
NN_PARAMS = dict(
    hidden_dims   = [256, 128, 64, 32],
    dropout_rates = [0.4, 0.3, 0.2, 0.1],
    batch_norm    = True,
    learning_rate = 0.0005,
    weight_decay  = 1e-4,
    batch_size    = 256,
    max_epochs    = 100,
    t0            = 10,
    t_mult        = 2,
    eta_min       = 1e-6,
)

# ── CV settings ──────────────────────────────────────────────────
CV_SEEDS  = [42, 7, 13, 99, 2024]
CV_FOLDS  = 5

# ── Conformal prediction hyperparameters ─────────────────────────
ALPHA_CP   = 0.10                                       # guarantees ≥90% coverage
ALPHA_SENS = [0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.80, 0.90, 0.95]     # sensitivity sweep
ALPHA_SENS_GATE = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]

print('✅ Configuration set.')
print(f'   NN arch:        {NN_PARAMS["hidden_dims"]} | dropout={NN_PARAMS["dropout_rates"]}')
print(f'   Batch norm:     {NN_PARAMS["batch_norm"]}  |  epochs={NN_PARAMS["max_epochs"]}  |  lr={NN_PARAMS["learning_rate"]}')
print(f'   GAMMA={GAMMA}  BETA_KL={BETA_KL}  MU_COV={MU_COV}  EPS={EPS}  T={T}')
print(f'   CV: {len(CV_SEEDS)} seeds × {CV_FOLDS} folds = {len(CV_SEEDS)*CV_FOLDS} total runs per model')
print(f'   CP: alpha={ALPHA_CP}  |  sensitivity sweep (ML): {ALPHA_SENS}| sensitivity sweep (Gate): {ALPHA_SENS_GATE}')

## 🧹 Cell 4 — Preprocessing & Feature Engineering

In [ ]:
print('Preprocessing...')
df = df_raw.copy()

if 'subject_id' not in df.columns:
    raise ValueError('subject_id column required for subject-level split')

# Gender encoding
if 'gender' in df.columns and df['gender'].dtype == object:
    df['gender'] = (df['gender'] == 'M').astype(int)

# Age filter
df = df[df['age_years'] >= 18].reset_index(drop=True)
print(f'  After age filter (≥18): {len(df):,} rows')

# Urine output clip
if 'urineoutput_24h' in df.columns:
    df['urineoutput_24h'] = df['urineoutput_24h'].clip(upper=20000)

subject_ids = df['subject_id'].values
for c in ['subject_id', 'hadm_id', 'icustay_id']:
    if c in df.columns:
        df = df.drop(columns=[c])

# Drop specified columns
COLS_TO_DROP = [c for c in df.columns
                if (c.startswith('diabp_') or c.startswith('bun_'))
                and '_was_missing' not in c]
df = df.drop(columns=COLS_TO_DROP, errors='ignore')
if COLS_TO_DROP:
    print(f'  Dropped {len(COLS_TO_DROP)} diabp_/bun_ columns')

# KNN imputation if needed
if df.isnull().sum().sum() > 0:
    print(f'  {df.isnull().sum().sum():,} missing values — running KNN imputation...')
    non_feat = [LABEL_COL]
    feat_all = [c for c in df.columns if c not in non_feat]
    miss_rate = df[feat_all].isnull().mean()
    for col in miss_rate[miss_rate > 0.10].index:
        df[f'{col}_was_missing'] = df[col].isnull().astype(int)
    for col in miss_rate[miss_rate > 0.60].index:
        df[col] = df[col].fillna(df[col].median())
    knn_cols = miss_rate[(miss_rate > 0) & (miss_rate <= 0.60)].index.tolist()
    if knn_cols:
        sc_imp = StandardScaler()
        X_sc   = sc_imp.fit_transform(df[knn_cols])
        X_imp  = KNNImputer(n_neighbors=5, weights='distance').fit_transform(X_sc)
        df[knn_cols] = sc_imp.inverse_transform(X_imp)
    print('  ✅ Imputation complete.')
else:
    print('  No missing values — skipping imputation.')

feature_names = [c for c in df.columns if c != LABEL_COL]
X_raw = df[feature_names].values.astype(float)
y     = df[LABEL_COL].values.astype(float)

N_NEG = int((y == 0).sum())
N_POS = int((y == 1).sum())
POS_WEIGHT = N_NEG / N_POS

print(f'\n✅ Final dataset: {X_raw.shape[0]:,} rows × {X_raw.shape[1]} features')
print(f'   Positive rate: {y.mean()*100:.2f}%  |  pos_weight: {POS_WEIGHT:.1f}x')

## 📏 Cell 5 — Human Expert Rule

In [ ]:
# Feature index lookup
MEANBP_IDX     = feature_names.index('meanbp_mean')
MEANBP_MIN_IDX = feature_names.index('meanbp_min')
SPO2_IDX       = feature_names.index('spo2_mean')
SPO2_MIN_IDX   = feature_names.index('spo2_min')
GCS_MEAN_IDX   = feature_names.index('gcs_mean')
BICARB_IDX     = feature_names.index('bicarb_mean')
RR_IDX         = feature_names.index('rr_mean')
SYSBP_IDX      = feature_names.index('sysbp_mean')
URINE_IDX      = feature_names.index('urineoutput_24h')

def human_rule(X):
    """
    g(x) in {-1, 0, 1}  — data-driven from EDA on label_24h_icu_mortality (base=2%).

    HIGH RISK (predict 1) — combined: mort=0.236 (11.7x base), recall=59.2%, coverage=5.1%
      A: sysbp_mean <= 80                              (extreme hypotension)
      B: spo2_min <= 85 AND bicarb_mean <= 14          (hypoxia + acidosis)
      C: mbp_min <= 45 AND spo2_min <= 85              (haemodynamic + hypoxic collapse)
      D: spo2_min <= 85 AND urine <= 300               (hypoxia + oliguria)
      E: mbp_min <= 45 AND urine <= 300                (hypotension + oliguria)
      F: bicarb_mean <= 12                             (metabolic acidosis)
      G: spo2_mean <= 90 AND bicarb_mean <= 15         (severe hypoxia + acidosis)
      H: gcs_mean <= 5 AND mbp_mean < 65               (coma + hypotension)
      I: gcs_mean <= 8 AND urine <= 300                (obtunded + oliguria)
      J: sysbp_mean <= 85 AND gcs_mean <= 8            (hypotension + obtunded)

    LOW RISK (predict 0) — mort=0.002 (0.1x base), coverage=35.3%
      spo2>=95 AND gcs>=13 AND mbp>=65 AND bicarb>=20 AND urine>500
    """
    g      = np.full(len(X), -1.0)
    sysbp  = X[:, SYSBP_IDX]
    spo2m  = X[:, SPO2_MIN_IDX]
    mbpm   = X[:, MEANBP_MIN_IDX]
    bicarb = X[:, BICARB_IDX]
    urine  = X[:, URINE_IDX]
    spo2   = X[:, SPO2_IDX]
    gcs    = X[:, GCS_MEAN_IDX]
    mbp    = X[:, MEANBP_IDX]

    valid  = (spo2 > 0) & (gcs > 0) & (mbp > 0)

    high_A = valid & (sysbp > 0)    & (sysbp <= 80)
    high_B = valid & (spo2m <= 85)  & (bicarb <= 14) & (bicarb > 0)
    high_C = valid & (mbpm  <= 45)  & (spo2m <= 85)
    high_D = valid & (spo2m <= 85)  & (urine > 0)    & (urine <= 300)
    high_E = valid & (mbpm  <= 45)  & (urine > 0)    & (urine <= 300)
    high_F = valid & (bicarb <= 12) & (bicarb > 0)
    high_G = valid & (spo2  <= 90)  & (bicarb <= 15) & (bicarb > 0)
    high_H = valid & (gcs   <= 5)   & (mbp < 65)
    high_I = valid & (gcs   <= 8)   & (urine > 0)    & (urine <= 300)
    high_J = valid & (sysbp > 0)    & (sysbp <= 85)  & (gcs <= 8)

    g[high_A | high_B | high_C | high_D | high_E |
      high_F | high_G | high_H | high_I | high_J] = 1.0

    low = (valid & (spo2 >= 95) & (gcs >= 13) & (mbp >= 65)
           & (bicarb >= 20) & (urine > 500))
    g[low & (g == -1)] = 0.0
    return g

# Quick sanity check on full dataset
g_all = human_rule(X_raw)
print('✅ Human rule defined.')
print(f'   Full dataset coverage:')
print(f'     Abstain (-1):  {(g_all==-1).sum():,} ({(g_all==-1).mean()*100:.1f}%)')
print(f'     High-risk (1): {(g_all==1).sum():,}  ({(g_all==1).mean()*100:.1f}%)  '
      f'mort={y[g_all==1].mean():.3f}' if (g_all==1).any() else '')
print(f'     Low-risk  (0): {(g_all==0).sum():,}  ({(g_all==0).mean()*100:.1f}%)  '
      f'mort={y[g_all==0].mean():.3f}' if (g_all==0).any() else '')

## 🧠 Cell 6 — Neural Network Architecture & Training

In [ ]:
class MortalityNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout_rates, batch_norm):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h_dim, drop in zip(hidden_dims, dropout_rates):
            layers.append(nn.Linear(in_dim, h_dim))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(drop))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))


def train_nn(Xtr, ytr, pw, input_dim, seed=42):
    torch.manual_seed(seed)
    p         = NN_PARAMS
    model     = MortalityNet(input_dim, p['hidden_dims'], p['dropout_rates'], p['batch_norm']).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], dtype=torch.float32).to(DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=p['learning_rate'], weight_decay=p['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                    optimizer, T_0=p['t0'], T_mult=p['t_mult'], eta_min=p['eta_min'])
    loader    = DataLoader(
                    TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                                  torch.tensor(ytr, dtype=torch.float32)),
                    batch_size=p['batch_size'], shuffle=True, drop_last=False)
    for _ in range(p['max_epochs']):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
    model.eval()
    return model


def nn_predict(model, X_np):
    model.eval()
    with torch.no_grad():
        return model.predict_proba(
            torch.tensor(X_np, dtype=torch.float32).to(DEVICE)
        ).cpu().numpy()


print('✅ MortalityNet defined.')
# Quick architecture summary
p = NN_PARAMS
print(f'   Architecture: {X_raw.shape[1]} → {" → ".join(map(str, p["hidden_dims"]))} → 1')
print(f'   Dropout:      {p["dropout_rates"]}')
print(f'   Batch norm:   {p["batch_norm"]}  |  Activation: GELU')
print(f'   Optimiser:    AdamW  |  LR={p["learning_rate"]}  |  WD={p["weight_decay"]}')
print(f'   Scheduler:    CosineAnnealingWarmRestarts (T0={p["t0"]}, Tmult={p["t_mult"]})')

## 🔀 Cell 7 — MoE Utilities & Training Functions

In [ ]:
# ── Helpers ──────────────────────────────────
def sigmoid(z):   return expit(np.clip(z, -30, 30))
def add_bias(X):  return np.hstack([np.ones((len(X), 1)), X])

def weighted_nll(p, y, pw):
    p = np.clip(p, 1e-7, 1-1e-7)
    w = np.where(y == 1, pw, 1.0)
    return -np.mean(w * (y * np.log(p) + (1-y) * np.log(1-p)))

def moe_predict_nn(f_nn, w, g, bX):
    rho   = sigmoid(bX @ w)
    f     = np.clip(f_nn, 1e-7, 1-1e-7)
    known = g != -1
    yh    = f.copy()
    yh[known] = (1 - rho[known]) * f[known] + rho[known] * g[known]
    return np.clip(yh, 1e-7, 1-1e-7)

def moe_loss_nn(w, f_nn, y, g, bX, gamma, pw):
    return weighted_nll(moe_predict_nn(f_nn, w, g, bX), y, pw) + gamma * np.sum(np.abs(w))

def moe_loss_kl_nn(w, f_nn, y, g, bX, gamma, beta, mu, pw):
    yh    = moe_predict_nn(f_nn, w, g, bX)
    nll   = weighted_nll(yh, y, pw)
    reg   = gamma * np.sum(np.abs(w))
    known = g != -1
    if not known.any():
        return nll + reg
    p_g = np.clip(g[known], 1e-7, 1-1e-7)
    q   = np.clip(yh[known], 1e-7, 1-1e-7)
    kl  = p_g * np.log(p_g/q) + (1-p_g) * np.log((1-p_g)/(1-q))
    rho = np.clip(sigmoid(bX[known] @ w), 1e-7, 1-1e-7)
    return nll + reg + beta * kl.mean() + (-mu * np.log(rho).mean())

def soft_cov(w, bX, g):
    known = g != -1
    return float(np.mean(sigmoid(bX[known] @ w))) * 100 if known.any() else 0.0

def brier_score(y_true, y_prob):
    return float(np.mean((y_prob - y_true) ** 2))

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece  = 0.0
    n    = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece

print('✅ Helper functions defined!')

# ── MoE trainers ─────────────────────────────
def train_std_moe_nn(f_nn_tr, ytr, gtr, bXtr, gamma, pw):
    d = bXtr.shape[1]; best, bw = np.inf, None
    for _ in range(N_RESTARTS):
        res = minimize(
            lambda w: moe_loss_nn(w, f_nn_tr, ytr, gtr, bXtr, gamma, pw),
            np.random.randn(d) * 0.01, method='L-BFGS-B',
            options={'maxiter': 300, 'ftol': 1e-8})
        if res.fun < best:
            best = res.fun; bw = res.x
    return bw, best

def train_pref_lb_nn(f_nn_tr, ytr, gtr, bXtr, w_s, L_star, gamma, eps, t, pw):
    d = bXtr.shape[1]; budget = (1 + eps) * L_star
    def lb(w):
        L   = moe_loss_nn(w, f_nn_tr, ytr, gtr, bXtr, gamma, pw)
        bar = budget - L
        rho = np.clip(sigmoid(bXtr @ w), 1e-7, 1-1e-7)
        return 1e10 if bar <= 0 else -t * np.sum(np.log(rho)) - np.log(bar)
    best, bw = np.inf, None
    for _ in range(N_RESTARTS):
        res = minimize(lb, w_s + np.random.randn(d) * 0.01,
                       method='L-BFGS-B', options={'maxiter': 500, 'ftol': 1e-9})
        w_c = res.x
        if moe_loss_nn(w_c, f_nn_tr, ytr, gtr, bXtr, gamma, pw) <= budget + 1e-5 and res.fun < best:
            best = res.fun; bw = w_c
    return bw if bw is not None else w_s.copy()

def train_pref_kl_nn(f_nn_tr, ytr, gtr, bXtr, w_s, gamma, beta, mu, pw):
    d = bXtr.shape[1]; best, bw = np.inf, None
    for _ in range(N_RESTARTS):
        res = minimize(
            lambda w: moe_loss_kl_nn(w, f_nn_tr, ytr, gtr, bXtr, gamma, beta, mu, pw),
            w_s + np.random.randn(d) * 0.01,
            method='L-BFGS-B', options={'maxiter': 500, 'ftol': 1e-9})
        if np.isfinite(res.fun) and res.fun < best:
            best = res.fun; bw = res.x
    return bw if bw is not None else w_s.copy()

print('✅ Model trainers defined!')

# ── Conformal Prediction on Gating Function ──────────────────────────────────
class ConformalGating:
    """
    Split conformal prediction on the gating function.

    The gating function is treated as a binary classifier:
      - class 1 = "follow human rule"
      - class 0 = "use ML"

    True gating labels for calibration are derived from which expert
    was actually correct on the calibration set:
      - If human rule correct               -> true label = 1 (follow human)
      - If human rule wrong, ML correct     -> true label = 0 (use ML)
      - If both correct                     -> true label = 1 (preferential philosophy)
      - If both wrong                       -> true label = 1 (no reason to override)

    Nonconformity score: s = 1 - p_hat(true gating label).
    """

    def __init__(self, alpha=0.10):
        self.alpha     = alpha
        self.threshold = None

    def _derive_gating_labels(self, g_cal, y_cal, ml_preds_cal):
        """
        Derive the 'true' gating label for each calibration patient.

        Parameters
        ----------
        ml_preds_cal : hard ML predictions (0 or 1), shape (n_cal,)
                       computed by the caller before passing in

        Returns
        -------
        gating_labels : array, shape (n_cal,)
            1.0  = follow human rule
            0.0  = use ML
           -1.0  = human rule not applicable (g == -1)
        """
        gating_labels = np.full(len(g_cal), -1.0)
        for i in range(len(g_cal)):
            if g_cal[i] == -1:
                gating_labels[i] = -1
                continue
            human_correct = (g_cal[i] == y_cal[i])
            ml_correct    = (ml_preds_cal[i] == y_cal[i])
            if human_correct:
                gating_labels[i] = 1.0
            elif ml_correct:
                gating_labels[i] = 0.0
            else:
                gating_labels[i] = 1.0   # both wrong — no reason to override human
        return gating_labels

    def calibrate(self, w, bX_cal, g_cal, y_cal, ml_preds_cal):
        """
        Compute nonconformity scores on the calibration set and set threshold.

        Parameters
        ----------
        w            : gating weight vector
        bX_cal       : bias-augmented scaled features, shape (n_cal, d+1)
        g_cal        : human rule outputs in {-1, 0, 1}, shape (n_cal,)
        y_cal        : true labels, shape (n_cal,)
        ml_preds_cal : hard ML predictions (0 or 1) computed by caller,
                       shape (n_cal,)
        """
        rho           = np.clip(sigmoid(bX_cal @ w), 1e-7, 1 - 1e-7)
        gating_labels = self._derive_gating_labels(g_cal, y_cal, ml_preds_cal)

        valid = gating_labels != -1
        if valid.sum() == 0:
            self.threshold = 1.0
            return self

        rho_valid    = rho[valid]
        labels_valid = gating_labels[valid]

        # s = 1 - p_hat(true gating label)
        scores = np.where(
            labels_valid == 1,
            1 - rho_valid,   # true = "follow human": confidence is rho
            rho_valid         # true = "use ML":       confidence is 1 - rho
        )
        
        self.scores = scores  
        n       = len(scores)
        q_level = np.ceil((1 - self.alpha) * (n + 1)) / n
        self.threshold = np.quantile(scores, min(q_level, 1.0))
        return self

    def predict_sets(self, w, bX_test, g_test):
        """
        Produce conformal prediction sets for the gating decision.

        Returns
        -------
        gating_sets : list of sets, length n_test
            {1}    = confidently follow human rule
            {0}    = confidently use ML
            {0, 1} = uncertain
        """
        rho         = np.clip(sigmoid(bX_test @ w), 1e-7, 1 - 1e-7)
        gating_sets = []
        for i in range(len(g_test)):
            if g_test[i] == -1:
                gating_sets.append({0})
                continue
            prediction_set = set()
            if (1 - rho[i]) <= self.threshold:
                prediction_set.add(1)
            if rho[i] <= self.threshold:
                prediction_set.add(0)
            if len(prediction_set) == 0:
                if rho[i] >= 0.5:
                    prediction_set = {1}
                else:
                    prediction_set = {0}
            gating_sets.append(prediction_set)
        return gating_sets
print('✅ ConformalGating defined!')

# ── Conformal Prediction on ML (NN) Classifier ───────────────────────────────
class ConformalMLClassifier:
    """
    Split conformal prediction on the neural network classifier.
    Nonconformity score: s = 1 - p_hat(true label).
    Calibration uses pre-computed NN probabilities directly.
    """

    def __init__(self, alpha=0.10):
        self.alpha     = alpha
        self.threshold = None

    def calibrate(self, f_nn_cal, y_cal):
        """
        Parameters
        ----------
        f_nn_cal : NN predicted probabilities on calibration set, shape (n_cal,)
        y_cal    : true labels, shape (n_cal,)
        """
        probs  = np.clip(f_nn_cal, 1e-7, 1 - 1e-7)
        scores = np.where(y_cal == 1, 1 - probs, probs)
        self.scores = scores 
        n      = len(scores)
        q_level = np.ceil((1 - self.alpha) * (n + 1)) / n
        self.threshold = np.quantile(scores, min(q_level, 1.0))
        return self

    def predict_sets(self, f_nn_test):
        """
        Parameters
        ----------
        f_nn_test : NN predicted probabilities on test set, shape (n_test,)

        Returns
        -------
        ml_sets : list of sets, length n_test
            {1}    = confidently positive (high mortality risk)
            {0}    = confidently negative (low mortality risk)
            {0, 1} = uncertain
        """
        probs   = np.clip(f_nn_test, 1e-7, 1 - 1e-7)
        ml_sets = []
        for i in range(len(probs)):
            prediction_set = set()
            if (1 - probs[i]) <= self.threshold:
                prediction_set.add(1)
            if probs[i] <= self.threshold:
                prediction_set.add(0)
            if len(prediction_set) == 0:
                if probs[i] >= 0.5:
                    prediction_set = {1}
                else:
                    prediction_set = {0}
            ml_sets.append(prediction_set)
        return ml_sets

print('✅ ConformalMLClassifier defined!')

# ── Combined Conformal Deferral System ───────────────────────────────────────
class ConformalDeferralSystem:
    """
    Combines CP on the gating function and CP on the NN classifier into a
    four-outcome decision system.

    Outcomes
    --------
    follow_human   : gating CP set = {1}
    use_ml         : gating selects ML AND ML CP set is a singleton
    fallback_human : gating selects ML BUT ML is uncertain, and human rule exists
    escalate       : gating uncertain ({0,1}), OR ML uncertain with no human rule

    """

    def __init__(self, alpha_gating=0.10, alpha_ml=0.10,
                 cp_gating_on=True, cp_ml_on=True):
        self.cp_gating_on = cp_gating_on
        self.cp_ml_on     = cp_ml_on
        self.cp_gating    = ConformalGating(alpha=alpha_gating)   if cp_gating_on else None
        self.cp_ml        = ConformalMLClassifier(alpha=alpha_ml) if cp_ml_on     else None

    def calibrate(self, w, bX_cal, g_cal, y_cal, f_nn_cal):
        """
        Fit CP thresholds on a held-out calibration set.

        Parameters
        ----------
        w        : gating weight vector
        bX_cal   : bias-augmented scaled features, shape (n_cal, d+1)
        g_cal    : human rule outputs in {-1, 0, 1}, shape (n_cal,)
        y_cal    : true labels, shape (n_cal,)
        f_nn_cal : NN predicted probabilities on calibration set, shape (n_cal,)
        """
        # Compute hard ML predictions here and pass to gating calibrator,
        # matching the diabetes pattern where the caller derives ml_preds
        ml_preds = (f_nn_cal >= 0.5).astype(float)
        if self.cp_gating is not None:
            self.cp_gating.calibrate(w, bX_cal, g_cal, y_cal, ml_preds)
        if self.cp_ml is not None:
            self.cp_ml.calibrate(f_nn_cal, y_cal)
        return self

    def _summary(self, outcomes):
        unique, counts = np.unique(outcomes, return_counts=True)
        summary = dict(zip(unique, counts))
        total   = len(outcomes)
        return {
            'counts': summary,
            'rates':  {k: v / total * 100 for k, v in summary.items()},
            'total':  total,
        }

    def predict(self, w, bX_test, g_test, f_nn_test):
        """
        Route each patient to one of four outcomes.

        Parameters
        ----------
        w         : gating weight vector
        bX_test   : bias-augmented scaled features, shape (n_test, d+1)
        g_test    : human rule outputs in {-1, 0, 1}, shape (n_test,)
        f_nn_test : NN predicted probabilities on test set, shape (n_test,)

        Returns
        -------
        outcomes    : array of strings, shape (n_test,)
        predictions : array of floats in (0, 1), shape (n_test,)
        summary     : dict — {'counts': {...}, 'rates': {...}, 'total': N}
        ml_sets     : list of sets from ConformalMLClassifier, or None
        """
        n           = len(g_test)
        outcomes    = np.array([''] * n, dtype=object)
        predictions = np.full(n, np.nan)
        ml_probs    = np.clip(f_nn_test, 1e-7, 1 - 1e-7)
        rho         = sigmoid(bX_test @ w)

        remaining_idx = np.arange(n)

        # Pre-compute prediction sets only for active components
        gating_sets = None
        ml_sets     = None
        if self.cp_gating is not None:
            gating_sets = self.cp_gating.predict_sets(w, bX_test, g_test)
        if self.cp_ml is not None:
            ml_sets = self.cp_ml.predict_sets(f_nn_test)

        for i in remaining_idx:

            # ── Gating decision ───────────────────────────────────────────────
            if g_test[i] == -1:
                gating_decision = 'use_ml'
            elif self.cp_gating_on and gating_sets is not None:
                g_set = gating_sets[i]
                if g_set == {1}:
                    gating_decision = 'follow_human'
                elif g_set == {0, 1}:
                    gating_decision = 'escalate'
                else:
                    gating_decision = 'use_ml'
            else:
                # CP gating OFF — use raw rho threshold
                if rho[i] >= 0.5:
                    gating_decision = 'follow_human'
                else:
                    gating_decision = 'use_ml'

            # ── Act on gating decision ────────────────────────────────────────
            if gating_decision == 'follow_human':
                outcomes[i]    = 'follow_human'
                predictions[i] = float(g_test[i])
                continue

            if gating_decision == 'escalate':
                outcomes[i]    = 'escalate'
                predictions[i] = 0.5
                continue

            # ── ML decision ───────────────────────────────────────────────────
            if self.cp_ml_on and ml_sets is not None:
                m_set = ml_sets[i]
                if len(m_set) == 1:
                    outcomes[i]    = 'use_ml'
                    predictions[i] = ml_probs[i]
                else:
                    if g_test[i] != -1:
                        outcomes[i]    = 'fallback_human'
                        predictions[i] = float(g_test[i])
                    else:
                        outcomes[i]    = 'escalate'
                        predictions[i] = 0.5
            else:
                # CP ML OFF — use NN prediction directly
                outcomes[i]    = 'use_ml'
                predictions[i] = ml_probs[i]

        return outcomes, predictions, self._summary(outcomes), ml_sets, gating_sets

# ── Evaluation helpers ────────────────────────────────────────────────────────
def compute_ml_accuracy(outcomes, predictions, y_true):
    """
    Among patients routed to use_ml, fraction predicted correctly
    at hard threshold 0.5. 
    """
    ml_mask = (outcomes == 'use_ml')
    if ml_mask.sum() == 0:
        return np.nan
    ml_preds_hard = (predictions[ml_mask] >= 0.5).astype(float)
    return (ml_preds_hard == y_true[ml_mask]).mean()

def compute_conformal_coverage(ml_sets, outcomes, y_true, gating_sets=None):
    if gating_sets is not None:
        # Coverage = fraction not escalated (works for gating-only and both)
        mask = np.isin(outcomes, ['follow_human', 'use_ml', 'fallback_human'])
        return mask.sum() / len(outcomes)
    if ml_sets is not None:
        mask = np.isin(outcomes, ['use_ml', 'fallback_human'])
        if mask.sum() == 0:
            return np.nan
        covered = sum(y_true[i] in ml_sets[i] for i in np.where(mask)[0])
        return covered / mask.sum()
    return np.nan

def record_cp_model(key, outcomes, predictions, ml_sets, y_test,
                    w, bX_test, g_test, summary, gating_sets=None):
    """
    Record all metrics for one CP model on one fold into records[key].
    """
    pred_filled = np.where(np.isnan(predictions), 0.5, predictions)

    auc_all = roc_auc_score(y_test, pred_filled) \
              if len(np.unique(y_test)) > 1 else float('nan')
    non_esc = ~np.isin(outcomes, ['escalate'])
    auc_ne  = (roc_auc_score(y_test[non_esc], predictions[non_esc])
               if (non_esc.sum() > 0 and len(np.unique(y_test[non_esc])) > 1)
               else np.nan)

    esc_rate = summary['rates'].get('escalate', 0.0)   # no ood_abstain term
    ml_acc   = compute_ml_accuracy(outcomes, predictions, y_test)
    cp_cov = compute_conformal_coverage(ml_sets, outcomes, y_test,
                                            gating_sets=gating_sets) \
                if (ml_sets is not None or gating_sets is not None) else np.nan

    bs_all = brier_score(y_test, pred_filled)
    bs_ne  = brier_score(y_test[non_esc], predictions[non_esc]) \
             if non_esc.sum() > 0 else np.nan
    ece_all = expected_calibration_error(y_test, pred_filled)
    ece_ne  = expected_calibration_error(y_test[non_esc], predictions[non_esc]) \
              if non_esc.sum() > 0 else np.nan

    records[key]['auc'].append(auc_all)
    records[key]['auc_non_abstain'].append(auc_ne)
    records[key]['cov'].append(soft_cov(w, bX_test, g_test))
    records[key]['eff_cov'].append(np.mean(outcomes == 'follow_human') * 100)
    records[key]['abstain_rate'].append(0.0)             
    records[key]['escalation_rate'].append(esc_rate)
    records[key]['ml_accuracy'].append(ml_acc)
    records[key]['conformal_coverage'].append(cp_cov)
    records[key]['brier'].append(bs_all)
    records[key]['brier_non_esc'].append(bs_ne)
    records[key]['ece'].append(ece_all)
    records[key]['ece_non_esc'].append(ece_ne)
    for outcome, count in summary['counts'].items():
        outcome_counts_all[key][outcome] = \
            outcome_counts_all[key].get(outcome, 0) + count

def hard_coverage_curve_nn(w, bX, g, f_nn, y, n_points=20):
    """
    Accuracy-coverage curve for NN-based MoE.
    Sweeps the gating threshold from 0 to 1, computing what fraction
    of patients follow the human rule (coverage) and the overall
    accuracy at each threshold.
    Used for Figure 1 Panel G — replicates Pradier et al. Figures 3 & 5.
    """
    thresholds = np.linspace(0.01, 0.99, n_points)
    rho        = sigmoid(bX @ w)
    ml_preds   = np.clip(f_nn, 1e-7, 1 - 1e-7)
    curve      = []

    for t in thresholds:
        # At this threshold, patients with rho >= t follow the human rule
        follow_human = (g != -1) & (rho >= t)
        use_ml       = ~follow_human

        preds = np.where(follow_human, g.astype(float), (ml_preds >= 0.5).astype(float))
        coverage = follow_human.mean() * 100
        accuracy = (preds == y).mean() * 100
        curve.append((coverage, accuracy))

    return curve
            
print('✅ ConformalDeferralSystem defined.')
print('   Outcomes: follow_human | use_ml | fallback_human | escalate')
print(f'   Default alpha: gating={ALPHA_CP}  ml={ALPHA_CP}')

## 🔁 Cell 8a — Setup & Baseline Loop (CV Training)

 5-Seed × 5-Fold Cross-Validation.

> ⏱️ Expected runtime: ~15–40 min depending on dataset size and GPU availability.
> Each of 25 folds trains a full NN (100 epochs) and fits 3 MoE variants.

In [ ]:
# ── Keys ──────────────────────────────────────────────────────────────────────
KEYS_BASE          = ['nn_only', 'std_moe', 'pref_lb', 'pref_kl']
KEYS_LB            = ['lb_cp_ml', 'lb_cp_gating', 'lb_cp_both']
KEYS_KL            = ['kl_cp_ml', 'kl_cp_gating', 'kl_cp_both']
KEYS               = KEYS_BASE + KEYS_LB + KEYS_KL
KEYS_WITH_OUTCOMES = KEYS_LB + KEYS_KL

CAL_FRACTION = 0.20   # fraction of training fold held out for CP calibration

# ── Results storage ───────────────────────────────────────────────────────────
# Original per-model lists 
MODEL_NAMES = [
    'Neural Network Only',
    'Standard MoE',
    'Preferential MoE (log-bar)',
    'Preferential MoE (KL-reg)',
]
all_results = {m: [] for m in MODEL_NAMES}
all_auprc   = {m: [] for m in MODEL_NAMES}
all_softcov = {m: [] for m in MODEL_NAMES}

# Extended records dict for CP ablations (8b, 8c)
METRIC_KEYS = ['auc', 'auc_non_abstain', 'cov', 'eff_cov',
               'abstain_rate', 'escalation_rate',
               'ml_accuracy', 'conformal_coverage',
               'brier', 'brier_non_esc',
               'ece',   'ece_non_esc']
records            = {k: {m: [] for m in METRIC_KEYS} for k in KEYS_BASE}
outcome_counts_all = {k: {} for k in KEYS_WITH_OUTCOMES}
stored_weights     = {}

In [ ]:
import pickle, os

# ── Load from cache if available ──────────────────────────────────

# 1. Determine the base directory
NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()

# 2. Define and create the weights directory
WEIGHTS_DIR = os.path.join(NOTEBOOK_DIR, 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True) 

# 3. Update the Cache Path to live inside the weights folder
CACHE_PATH = os.path.join(WEIGHTS_DIR, 'cell8a_cache.pkl')

# ── Load from cache if available ──────────────────────────────────
# Set to False so reviewers can use your provided results immediately
FORCE_RETRAIN = False 

if not FORCE_RETRAIN and os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        cache = pickle.load(f)
    stored_weights     = cache['stored_weights']
    records            = cache['records']
    outcome_counts_all = cache['outcome_counts_all']
    all_results        = cache['all_results']
    all_auprc          = cache['all_auprc']
    all_softcov        = cache['all_softcov']
    
    print(f'✅ Loaded Cell 8a cache from {CACHE_PATH} ({len(stored_weights)} entries)')
    print('   Skip to Cell 8b/8c — no need to re-run CV loop')
    # Note: SystemExit might restart the kernel in some environments; 
    # many researchers prefer using an 'if' block around the training code instead.
    raise SystemExit('Cache loaded — stopping cell execution')
else:
    print("🚀 Cache not found or FORCE_RETRAIN=True. Starting training loop...")

# ── CV loop ───────────────────────────────────────────────────────────────────
total_folds = len(CV_SEEDS) * CV_FOLDS
run_idx     = 0

print(f'Starting {len(CV_SEEDS)}-seed × {CV_FOLDS}-fold CV  ({total_folds} total runs)\n')
print(f'{"Seed":>5}  {"Fold":>4}  {"NN":>7}  {"StdMoE":>7}  {"PrefLB":>7}  {"PrefKL":>7}')
print('-' * 55)

for seed in CV_SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)

    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_raw, y), 1):
        run_idx += 1

        # ── Calibration split ─────────────────────────────────────────────────
        # Split training fold into train-proper + calibration set for CP fitting.
        # Scaler and NN are fitted on train-proper only; cal set is held out
        # exclusively for fitting CP thresholds in 8b and 8c.
        rs = seed * CV_FOLDS + fold_idx
        train_proper_idx, cal_idx = train_test_split(
            tr_idx, test_size=CAL_FRACTION, stratify=y[tr_idx], random_state=rs
        )

        Xtr_r,  Xval_r  = X_raw[train_proper_idx], X_raw[val_idx]
        Xcal_r          = X_raw[cal_idx]
        ytr,    yval    = y[train_proper_idx],      y[val_idx]
        ycal            = y[cal_idx]

        # Human rule
        gtr  = human_rule(Xtr_r)
        gcal = human_rule(Xcal_r)
        gval = human_rule(Xval_r)

        # Pos weight
        pw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)

        # Scale
        sc    = StandardScaler()
        Xtr   = sc.fit_transform(Xtr_r)
        Xcal  = sc.transform(Xcal_r)
        Xval  = sc.transform(Xval_r)
        bXtr  = add_bias(Xtr)
        bXcal = add_bias(Xcal)
        bXval = add_bias(Xval)
        input_dim = Xtr.shape[1]

        # ── Train NN ──────────────────────────────────────────────────────────
        nn_model  = train_nn(Xtr, ytr, pw, input_dim, seed=seed + fold_idx)
        f_nn_tr   = nn_predict(nn_model, Xtr)
        f_nn_cal  = nn_predict(nn_model, Xcal)
        f_nn_val  = nn_predict(nn_model, Xval)

        # ── Standard MoE ──────────────────────────────────────────────────────
        w_s, L_star = train_std_moe_nn(f_nn_tr, ytr, gtr, bXtr, GAMMA, pw)
        p_std       = moe_predict_nn(f_nn_val, w_s, gval, bXval)

        # ── Pref MoE (log-barrier) ────────────────────────────────────────────
        w_p    = train_pref_lb_nn(f_nn_tr, ytr, gtr, bXtr, w_s, L_star, GAMMA, EPS, T, pw)
        p_pref = moe_predict_nn(f_nn_val, w_p, gval, bXval)

        # ── Pref MoE (KL) ────────────────────────────────────────────────────
        np.random.seed(seed * 1000 + fold_idx)
        w_k  = train_pref_kl_nn(f_nn_tr, ytr, gtr, bXtr, w_s, GAMMA, BETA_KL, MU_COV, pw)
        p_kl = moe_predict_nn(f_nn_val, w_k, gval, bXval)

        # ── Score ─────────────────────────────────────────────────────────────
        skip    = len(np.unique(yval)) < 2
        preds   = [f_nn_val, p_std,  p_pref, p_kl]
        weights = [None,     w_s,    w_p,    w_k ]

        row_aucs = []
        for name, pred, w in zip(MODEL_NAMES, preds, weights):
            auc    = roc_auc_score(yval, pred)           if not skip else float('nan')
            auprc  = average_precision_score(yval, pred) if not skip else float('nan')
            sc_val = soft_cov(w, bXval, gval)            if w is not None else 0.0
            all_results[name].append(auc)
            all_auprc[name].append(auprc)
            all_softcov[name].append(sc_val)
            row_aucs.append(auc)

        # Also populate records dict (used by 8b/8c results tables)
        for key, pred, w in zip(KEYS_BASE, preds, weights):
            auc    = roc_auc_score(yval, pred)           if not skip else float('nan')
            sc_val = soft_cov(w, bXval, gval)            if w is not None else 0.0
            records[key]['auc'].append(auc)
            records[key]['auc_non_abstain'].append(auc)
            records[key]['cov'].append(sc_val)
            records[key]['eff_cov'].append(float('nan'))
            records[key]['abstain_rate'].append(0.0)
            records[key]['escalation_rate'].append(0.0)
            records[key]['ml_accuracy'].append(float('nan'))
            records[key]['conformal_coverage'].append(float('nan'))
            records[key]['brier'].append(brier_score(yval, pred))
            records[key]['brier_non_esc'].append(brier_score(yval, pred))
            records[key]['ece'].append(expected_calibration_error(yval, pred))
            records[key]['ece_non_esc'].append(expected_calibration_error(yval, pred))

        # ── Store fold data for 8b and 8c ─────────────────────────────────────
        stored_weights[(seed, fold_idx)] = {
            'w_s':      w_s.copy(), 
            'w_p':      w_p.copy(),
            'w_k':      w_k.copy(),
            'bXcal':    bXcal.copy(),
            'bXval':    bXval.copy(),
            'gcal':     gcal.copy(),
            'gval':     gval.copy(),
            'ycal':     ycal.copy(),
            'yval':     yval.copy(),
            'f_nn_cal': f_nn_cal.copy(),
            'f_nn_val': f_nn_val.copy(),
        }

        print(f'  {seed:>5}  fold{fold_idx}  '
              f'{row_aucs[0]:.4f}   {row_aucs[1]:.4f}   {row_aucs[2]:.4f}   {row_aucs[3]:.4f}')

print('\n✅ Base CV complete!')
print(f'   stored_weights has {len(stored_weights)} entries — ready for 8b and 8c')

# ── Save cache ────────────────────────────────────────────────────
with open(CACHE_PATH, 'wb') as f:
    pickle.dump({
        'stored_weights':     stored_weights,
        'records':            records,
        'outcome_counts_all': outcome_counts_all,
        'all_results':        all_results,
        'all_auprc':          all_auprc,
        'all_softcov':        all_softcov,
    }, f)
print(f'✅ Cell 8a cache saved → {CACHE_PATH}')
print('   Next session: cache will be loaded automatically, skipping CV loop')

## 🔁 Cell 8b — Log-Barrier Ablations (CV Training)

In [ ]:
# ── Reinitialise LB records (safe to re-run without redoing 8a) ───────────────
for k in KEYS_LB:
    records[k]            = {m: [] for m in METRIC_KEYS}
    outcome_counts_all[k] = {}

print('Starting LB CP ablations ...')
print('(Each dot = 1 fold completed)\n')

for (seed, fold_idx), data in stored_weights.items():
    try:
        w_p      = data['w_p']
        bXcal    = data['bXcal'];    bXval    = data['bXval']
        gcal     = data['gcal'];     gval     = data['gval']
        ycal     = data['ycal'];     yval     = data['yval']
        f_nn_cal = data['f_nn_cal']
        f_nn_val = data['f_nn_val']

        # 1. LB + CP on ML only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=False, cp_ml_on=True)
        cds.calibrate(w_p, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_p, bXval, gval, f_nn_val)
        record_cp_model('lb_cp_ml', out, pred, ml_sets, yval, w_p, bXval, gval, summ, gating_sets=gating_sets)

        # 2. LB + CP on gating only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=False)
        cds.calibrate(w_p, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_p, bXval, gval, f_nn_val)
        record_cp_model('lb_cp_gating', out, pred, ml_sets, yval, w_p, bXval, gval, summ, gating_sets=gating_sets)

        # 3. LB + CP on both
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=True)
        cds.calibrate(w_p, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_p, bXval, gval, f_nn_val)
        record_cp_model('lb_cp_both', out, pred, ml_sets, yval, w_p, bXval, gval, summ, gating_sets=gating_sets)

        print('.', end='', flush=True)

    except Exception as e:
        print(f'\nERROR seed={seed} fold={fold_idx}: {e}')
        import traceback; traceback.print_exc()

print('\n\n✅ LB CP ablations complete!')

## 🔁 Cell 8c — KL-Regulatisation Ablations (CV Training)

In [ ]:
# ── Reinitialise KL records (safe to re-run without redoing 8a or 8b) ─────────
for k in KEYS_KL:
    records[k]            = {m: [] for m in METRIC_KEYS}
    outcome_counts_all[k] = {}

print('Starting KL CP ablations ...')
print('(Each dot = 1 fold completed)\n')

for (seed, fold_idx), data in stored_weights.items():
    try:
        w_k      = data['w_k']
        bXcal    = data['bXcal'];    bXval    = data['bXval']
        gcal     = data['gcal'];     gval     = data['gval']
        ycal     = data['ycal'];     yval     = data['yval']
        f_nn_cal = data['f_nn_cal']
        f_nn_val = data['f_nn_val']

        # 1. KL + CP on ML only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=False, cp_ml_on=True)
        cds.calibrate(w_k, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_k, bXval, gval, f_nn_val)
        record_cp_model('kl_cp_ml', out, pred, ml_sets, yval, w_k, bXval, gval, summ, gating_sets=gating_sets)

        # 2. KL + CP on gating only
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=False)
        cds.calibrate(w_k, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_k, bXval, gval, f_nn_val)
        record_cp_model('kl_cp_gating', out, pred, ml_sets, yval, w_k, bXval, gval, summ, gating_sets=gating_sets)

        # 3. KL + CP on both
        cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                                      cp_gating_on=True, cp_ml_on=True)
        cds.calibrate(w_k, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_k, bXval, gval, f_nn_val)
        record_cp_model('kl_cp_both', out, pred, ml_sets, yval, w_k, bXval, gval, summ, gating_sets=gating_sets)

        print('.', end='', flush=True)

    except Exception as e:
        print(f'\nERROR seed={seed} fold={fold_idx}: {e}')
        import traceback; traceback.print_exc()

print('\n\n✅ KL CP ablations complete!')

## 📊 Cell 9 — Final Results Summary

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 9 — Final Results Summary
# ════════════════════════════════════════════════════════════════════

def ci(arr):
    a = np.array(arr, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) == 0:
        return np.nan, np.nan, np.nan
    m, s, n = a.mean(), a.std(), len(a)
    return m, m - 1.96*s/np.sqrt(n), m + 1.96*s/np.sqrt(n)

def get_mean(key, metric):
    arr = np.array(records[key][metric], dtype=float)
    arr = arr[~np.isnan(arr)]
    return arr.mean() if len(arr) > 0 else np.nan

LABEL_MAP = {
    'nn_only':       'NN Only',
    'std_moe':       'Standard MoE',
    'pref_lb':       'Pref MoE (log-barrier)',
    'pref_kl':       'Pref MoE (KL-reg)',
    'lb_cp_ml':      'LB + CP-ML only',
    'lb_cp_gating':  'LB + CP-gating only',
    'lb_cp_both':    'LB + CP-both',
    'kl_cp_ml':      'KL + CP-ML only',
    'kl_cp_gating':  'KL + CP-gating only',
    'kl_cp_both':    'KL + CP-both',
}

# ════════════════════════════════════════════════════════════════════
# Table 1 — Predictive Performance
# ════════════════════════════════════════════════════════════════════
print('\n── Table 1: Predictive Performance ──')
print('=' * 118)
print(f"  {'Model':<38} {'AUC(all)':>8}  {'95% CI':>16}  {'AUC(non-esc)':>12}  {'Brier(all)':>10}  {'Brier(ne)':>9}  {'ECE(all)':>8}  {'ECE(ne)':>7}")
print('=' * 118)
for key in KEYS:
    am, al, ah = ci(records[key]['auc'])
    if np.isnan(am):
        print(f'  {LABEL_MAP[key]:<38}  N/A')
        continue
    na   = get_mean(key, 'auc_non_abstain')
    bs   = get_mean(key, 'brier')
    bsne = get_mean(key, 'brier_non_esc')
    ece  = get_mean(key, 'ece')
    ecene= get_mean(key, 'ece_non_esc')
    na_str    = f'{na:>12.4f}'   if not np.isnan(na)    else '         N/A'
    bs_str    = f'{bs:>10.4f}'   if not np.isnan(bs)    else '       N/A'
    bsne_str  = f'{bsne:>9.4f}'  if not np.isnan(bsne)  else '      N/A'
    ece_str   = f'{ece:>8.4f}'   if not np.isnan(ece)   else '     N/A'
    ecene_str = f'{ecene:>7.4f}' if not np.isnan(ecene) else '    N/A'
    print(f'  {LABEL_MAP[key]:<38} {am:>8.4f}  [{al:.4f}-{ah:.4f}]  {na_str}  {bs_str}  {bsne_str}  {ece_str}  {ecene_str}')
print('=' * 118)

# ════════════════════════════════════════════════════════════════════
# Table 2 — Safety Mechanism Behaviour
# ════════════════════════════════════════════════════════════════════
print('\n── Table 2: Safety Mechanism Behaviour ──')
print('=' * 107)
print(f"  {'Model':<38} {'SoftCov%':>9}  {'EffCov%':>8}  {'Esc%':>6}  {'ML Acc':>7}  {'CP Cov':>7}  {'Guarantee':>9}")
print('=' * 107)
for key in KEYS:
    am, *_ = ci(records[key]['auc'])
    if np.isnan(am):
        print(f'  {LABEL_MAP[key]:<38}  N/A')
        continue
    cm,  *_ = ci(records[key]['cov'])
    em,  *_ = ci(records[key]['eff_cov'])
    esm, *_ = ci(records[key]['escalation_rate'])
    abm, *_ = ci(records[key]['abstain_rate'])
    ml   = get_mean(key, 'ml_accuracy')
    cc   = get_mean(key, 'conformal_coverage')
    es_str      = f'{esm:>5.1f}%' if not np.isnan(esm) else '   N/A'
    ml_str   = f'{ml:>7.3f}'   if not np.isnan(ml)                  else '    N/A'
    cc_str   = f'{cc:>7.3f}'   if not np.isnan(cc)                  else '    N/A'
    guar_str = f'{(1-ALPHA_CP):>9.2f}' if key in KEYS_WITH_OUTCOMES else '      N/A'
    eff_str = f'{em:>7.1f}%' if not np.isnan(em) else '    N/A'
    print(f'  {LABEL_MAP[key]:<38} {cm:>8.1f}%  {eff_str}  {es_str}  {ml_str}  {cc_str}  {guar_str}')
print('=' * 107)
print(f'  Note: CP Cov should be ≥ {1-ALPHA_CP:.2f} (guarantee) for CP models')

# ════════════════════════════════════════════════════════════════════
# Table 3 — Ablation Summary
# ════════════════════════════════════════════════════════════════════
ablation_config = {
    'lb_cp_ml':      (True,  False),
    'lb_cp_gating':  (False, True),
    'lb_cp_both':    (True,  True),
    'kl_cp_ml':      (True,  False),
    'kl_cp_gating':  (False, True),
    'kl_cp_both':    (True,  True),
}

def print_ablation_row(key):
    cp_ml, cp_gate = ablation_config[key]
    esm, *_ = ci(records[key]['escalation_rate'])
    auc_ne   = get_mean(key, 'auc_non_abstain')
    cc       = get_mean(key, 'conformal_coverage')
    bsne     = get_mean(key, 'brier_non_esc')
    es_str   = f'{esm:>5.1f}%' if not np.isnan(esm) else '   N/A'
    auc_str  = f'{auc_ne:>8.4f}' if not np.isnan(auc_ne)            else '     N/A'
    cc_str   = f'{cc:>7.3f}'     if not np.isnan(cc)                else '    N/A'
    bsne_str = f'{bsne:>9.4f}'   if not np.isnan(bsne)              else '      N/A'
    cp_ml_str   = '  ✓  ' if cp_ml   else '  —  '
    cp_gate_str = '   ✓    ' if cp_gate else '   —    '
    print(f'  {LABEL_MAP[key]:<38} {cp_ml_str}  {cp_gate_str}  {es_str}  {auc_str}  {cc_str}  {bsne_str}')

print('\n── Table 3: Ablation Summary (CP models only) ──')
print('=' * 98)
print(f"  {'Model':<38} {'CP-ML':>6}  {'CP-Gate':>8}  {'Esc%':>6}  {'AUC(ne)':>8}  {'CP Cov':>7}  {'Brier(ne)':>9}")
print('=' * 98)
print('  — Log-Barrier base —')
for key in KEYS_LB:
    print_ablation_row(key)
print('  — KL-reg base —')
for key in KEYS_KL:
    print_ablation_row(key)
print('=' * 98)

# ════════════════════════════════════════════════════════════════════
# Experimental Configuration
# ════════════════════════════════════════════════════════════════════
print('\n── Experimental Configuration ──')
print(f'  CP α              = {ALPHA_CP}    — confidence level for conformal prediction')
print(f'  KL beta (β)       = {BETA_KL}     — weight of KL divergence term')
print(f'  KL mu (μ)         = {MU_COV}     — weight of coverage push term')
print(f'  Cal fraction      = {CAL_FRACTION}    — fraction of training fold held out for CP calibration')
print(f'  N seeds × N folds = {len(CV_SEEDS)} × {CV_FOLDS}    — total folds = {len(CV_SEEDS) * CV_FOLDS}')
print(f'  NN architecture   = {NN_PARAMS["hidden_dims"]}  drop={NN_PARAMS["dropout_rates"]}')

# ════════════════════════════════════════════════════════════════════
# Outcome Distributions
# ════════════════════════════════════════════════════════════════════
print('\n── Outcome Distributions (% of patients, accumulated across all folds) ──')

# ICU has 4 outcomes only — no ood_human or ood_abstain
outcome_order  = ['follow_human', 'use_ml', 'fallback_human', 'escalate']
LABEL_COL = 20
COL       = 16

# Group header row — LB / KL separator
lb_width = len(KEYS_LB) * (COL + 2)
kl_width = len(KEYS_KL) * (COL + 2)
group_header = (f"  {' ' * LABEL_COL}"
                f"  {'— Log-Barrier —':^{lb_width-2}}"
                f"  {'— KL-reg —':^{kl_width-2}}")
print(group_header)

# Column header row
header = f"  {'Outcome':<{LABEL_COL}}"
for key in KEYS_WITH_OUTCOMES:
    label = LABEL_MAP[key].replace('LB + ', '').replace('KL + ', '').replace(' only', '')
    header += f"  {label:^{COL}}"
print(header)

print('  ' + '-'*LABEL_COL + ('  ' + '-'*COL) * len(KEYS_WITH_OUTCOMES))

for outcome in outcome_order:
    row = f"  {outcome:<{LABEL_COL}}"
    has_any = False
    for key in KEYS_WITH_OUTCOMES:
        total = sum(outcome_counts_all[key].values())
        count = outcome_counts_all[key].get(outcome, 0)
        if total > 0 and count > 0:
            cell = f'{100 * count / total:.1f}%'
            row += f"  {cell:^{COL}}"
            has_any = True
        else:
            row += f"  {'—':^{COL}}"
    if has_any:
        print(row)

print('  ' + '-'*LABEL_COL + ('  ' + '-'*COL) * len(KEYS_WITH_OUTCOMES))
totals_row = f"  {'Total patients':<{LABEL_COL}}"
for key in KEYS_WITH_OUTCOMES:
    total = sum(outcome_counts_all[key].values())
    totals_row += f"  {str(total):^{COL}}"
print(totals_row)

print('  ' + '-'*LABEL_COL + ('  ' + '-'*COL) * len(KEYS_WITH_OUTCOMES))
row_h = f"  {'Handled (%)':<{LABEL_COL}}"
row_e = f"  {'Escalated (%)':<{LABEL_COL}}"
for key in KEYS_WITH_OUTCOMES:
    total = sum(outcome_counts_all[key].values())
    if total == 0:
        row_h += f"  {'N/A':^{COL}}"
        row_e += f"  {'N/A':^{COL}}"
        continue
    esc_count = outcome_counts_all[key].get('escalate', 0)
    row_h += f"  {f'{100*(total-esc_count)/total:.1f}%':^{COL}}"
    row_e += f"  {f'{100*esc_count/total:.1f}%':^{COL}}"
print(row_h)
print(row_e)

# ════════════════════════════════════════════════════════════════════
# Best CP Variant
# ════════════════════════════════════════════════════════════════════
# Definition: the best CP variant is the model that achieves the best
# AUC(non-esc) among models with a clinically meaningful escalation
# rate (5-15%). If no model falls in this range, we fall back to the
# model with the lowest escalation rate that still escalates at least
# one patient (i.e. CP is genuinely active), prioritising AUC(ne).
# Models with 0% escalation are excluded — they are equivalent to
# their base model and provide no additional safety benefit.
# ════════════════════════════════════════════════════════════════════

ESC_MIN =  5.0   # % — below this CP is not doing meaningful safety work
ESC_MAX = 15.0   # % — above this is clinically infeasible in ICU

def best_cp_variant(keys, records, label_map,
                    esc_min=ESC_MIN, esc_max=ESC_MAX):
    # Pass 1 — models within the clinically acceptable escalation range
    candidates = []
    for key in keys:
        esm, *_ = ci(records[key]['escalation_rate'])
        auc_ne   = get_mean(key, 'auc_non_abstain')
        if np.isnan(esm) or np.isnan(auc_ne):
            continue
        if esc_min <= esm <= esc_max:
            candidates.append((key, auc_ne, esm))

    if candidates:
        # Pick highest AUC(ne) within acceptable escalation range
        best = max(candidates, key=lambda x: x[1])
        note = f'(escalation within clinical target {esc_min}–{esc_max}%)'
    else:
        # Pass 2 — fall back to lowest escalation among genuinely active models
        candidates = []
        for key in keys:
            esm, *_ = ci(records[key]['escalation_rate'])
            auc_ne   = get_mean(key, 'auc_non_abstain')
            if np.isnan(esm) or np.isnan(auc_ne):
                continue
            if esm > 0:
                candidates.append((key, auc_ne, esm))
        if not candidates:
            print('  No active CP variants found — all models show 0% escalation')
            return
        # Among active models, pick lowest escalation then highest AUC(ne)
        best = min(candidates, key=lambda x: (x[2], -x[1]))
        note = f'(no model in {esc_min}–{esc_max}% range — lowest active escalation)'

    key, auc_ne, esm = best
    bsne = get_mean(key, 'brier_non_esc')
    ccne, *_ = ci(records[key]['conformal_coverage'])
    print(f'  {label_map[key]}')
    print(f'    AUC(non-esc): {auc_ne:.4f}  |  Esc: {esm:.1f}%  |  '
          f'Brier(ne): {bsne:.4f}  |  CP Cov: {ccne:.3f}')
    print(f'    {note}')

print('\n── Best CP Variant ──')
best_cp_variant(KEYS_WITH_OUTCOMES, records, LABEL_MAP)

## 📊 Cell 10 — Statistical Signifiance Testing

In [ ]:
from scipy.stats import wilcoxon, ttest_1samp

print('── Statistical Significance Testing (Wilcoxon Signed-Rank) ──\n')
print('  Baseline: Pref MoE (log-barrier) — original Pradier et al. model')
print('  H0: no difference in means | p < 0.05 = significant')
print('  CP models: CP-gating only tested (CP-ML inactive on MIMIC — 0% escalation)\n')

METRICS_A = [
    ('cov',             'Soft Cov%',    True),   # primary Innovation A claim
    ('auc_non_abstain', 'AUC(non-esc)', True),
    ('brier_non_esc',   'Brier(ne)',    False),
    ('ece_non_esc',     'ECE(ne)',      False),
]

METRICS_B = [
    ('auc_non_abstain', 'AUC(non-esc)', True),
    ('brier_non_esc',   'Brier(ne)',    False),
    ('ece_non_esc',     'ECE(ne)',      False),
]

col = 32
header = (f"  {'Model':<{col}} {'Metric':<16} {'Baseline':>9}  "
          f"{'Model':>9}  {'Diff':>8}  {'p-value':>9}  {'Sig':>4}")
divider = ('  ' + '-'*col + '  ' + '-'*15 + '  ' + '-'*9 +
           '  ' + '-'*9 + '  ' + '-'*8 + '  ' + '-'*9 + '  ' + '-'*4)

def run_wilcoxon_row(label, key, baseline_key, metric,
                     metric_label, higher_better, col):
    model_arr    = np.array(records[key][metric],          dtype=float)
    baseline_arr = np.array(records[baseline_key][metric], dtype=float)
    min_len      = min(len(model_arr), len(baseline_arr))
    model_arr    = model_arr[:min_len]
    baseline_arr = baseline_arr[:min_len]
    mask         = ~(np.isnan(model_arr) | np.isnan(baseline_arr))
    model_arr    = model_arr[mask]
    baseline_arr = baseline_arr[mask]
    if len(model_arr) < 10:
        print(f'  {label:<{col}} {metric_label:<16}  insufficient data')
        return
    diff = model_arr.mean() - baseline_arr.mean()
    try:
        _, p = wilcoxon(model_arr, baseline_arr)
    except ValueError:
        print(f'  {label:<{col}} {metric_label:<16}  no difference (p=1.000)')
        return
    sig       = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    direction = '↑' if (diff > 0 and higher_better) or \
                       (diff < 0 and not higher_better) else '↓'
    print(f'  {label:<{col}} {metric_label:<16} {baseline_arr.mean():>9.4f}  '
          f'{model_arr.mean():>9.4f}  {diff:>+8.4f}  {p:>9.4f}  {sig} {direction}')

# ════════════════════════════════════════════════════════════════════
# Innovation A — KL regularisation vs Log-barrier
# Primary claim: KL-reg improves soft coverage over log-barrier
# Known trade-off: KL degrades calibration (Brier/ECE) — expected
# ════════════════════════════════════════════════════════════════════
print('  — Innovation A: KL regularisation vs Log-barrier —')
print('  Primary claim: soft coverage improvement')
print('  Known trade-off: Brier/ECE degradation is expected from KL objective\n')
print(header)
print(divider)
for metric, metric_label, higher_better in METRICS_A:
    run_wilcoxon_row('Pref MoE (KL-reg)', 'pref_kl', 'pref_lb',
                     metric, metric_label, higher_better, col)
print()

# ════════════════════════════════════════════════════════════════════
# Innovation B — CP-gating safety mechanism
# Note: AUC(ne)/Brier(ne) comparisons are on different patient
# subsets after escalation — primary findings are in Table 2
# (escalation rate and CP coverage guarantee)
# ════════════════════════════════════════════════════════════════════
print('  — Innovation B: CP-gating safety mechanism —')
print('  Note: metrics computed on different subsets after escalation.')
print('  Primary Innovation B findings are escalation rate and CP coverage.\n')
print(header)
print(divider)

print('  vs Pref LB (does CP-gating add safety value over base LB?):')
for metric, metric_label, higher_better in METRICS_B:
    run_wilcoxon_row('LB + CP-gating only', 'lb_cp_gating', 'pref_lb',
                     metric, metric_label, higher_better, col)
print()

print('  vs Pref KL (does CP-gating add safety value over base KL?):')
for metric, metric_label, higher_better in METRICS_B:
    run_wilcoxon_row('KL + CP-gating only', 'kl_cp_gating', 'pref_kl',
                     metric, metric_label, higher_better, col)
print()

# ════════════════════════════════════════════════════════════════════
# Innovation B — CP coverage vs theoretical guarantee (1-α = 0.90)
# One-sample t-test against fixed threshold — tests whether each
# model statistically satisfies or violates the conformal guarantee
# ════════════════════════════════════════════════════════════════════
print('  — Innovation B: CP coverage vs theoretical guarantee —')
print(f'  One-sample t-test against guarantee threshold (1-α = {1-ALPHA_CP:.2f})\n')
print(f"  {'Model':<{col}} {'Metric':<16} {'Guarantee':>9}  "
      f"{'Empirical':>9}  {'Diff':>8}  {'p-value':>9}  {'Sig':>4}")
print(divider)

for key, label in [('lb_cp_gating', 'LB + CP-gating only'),
                   ('kl_cp_gating', 'KL + CP-gating only')]:
    cov_arr = np.array(records[key]['conformal_coverage'], dtype=float)
    cov_arr = cov_arr[~np.isnan(cov_arr)]
    if len(cov_arr) < 10:
        print(f'  {label:<{col}}  insufficient data')
        continue
    _, p      = ttest_1samp(cov_arr, 1 - ALPHA_CP)
    sig       = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    mean      = cov_arr.mean()
    diff      = mean - (1 - ALPHA_CP)
    direction = '↑ satisfies' if diff >= 0 else '↓ violates'
    print(f'  {label:<{col}} {"CP Cov":<16} {(1-ALPHA_CP):>9.3f}  '
          f'{mean:>9.3f}  {diff:>+8.3f}  {p:>9.4f}  {sig} {direction}')
print()

# ════════════════════════════════════════════════════════════════════
# Interaction — does KL-reg reduce CP-gating escalation burden?
# Tests whether Innovation A reduces the need for Innovation B to
# intervene — the core argument for combining both innovations
# ════════════════════════════════════════════════════════════════════
print('  — Interaction: does KL-reg reduce CP-gating escalation burden? —')
print('  Tests whether Innovation A reduces Innovation B intervention rate\n')
print(f"  {'Comparison':<{col}} {'Metric':<16} {'LB':>9}  "
      f"{'KL':>9}  {'Diff':>8}  {'p-value':>9}  {'Sig':>4}")
print(divider)

# Escalation rate
lb_esc = np.array(records['lb_cp_gating']['escalation_rate'], dtype=float)
kl_esc = np.array(records['kl_cp_gating']['escalation_rate'], dtype=float)
min_len = min(len(lb_esc), len(kl_esc))
lb_esc  = lb_esc[:min_len];  kl_esc = kl_esc[:min_len]
mask    = ~(np.isnan(lb_esc) | np.isnan(kl_esc))
lb_esc  = lb_esc[mask];      kl_esc = kl_esc[mask]
try:
    _, p  = wilcoxon(lb_esc, kl_esc)
    sig   = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    diff  = kl_esc.mean() - lb_esc.mean()
    direction = '↑' if diff < 0 else '↓'
    print(f'  {"KL vs LB CP-gating":<{col}} {"Esc Rate":<16} '
          f'{lb_esc.mean():>9.1f}  {kl_esc.mean():>9.1f}  '
          f'{diff:>+8.1f}  {p:>9.4f}  {sig} {direction}')
except ValueError:
    print('  No significant difference in escalation rates')

# ML accuracy
lb_ml = np.array(records['lb_cp_gating']['ml_accuracy'], dtype=float)
kl_ml = np.array(records['kl_cp_gating']['ml_accuracy'], dtype=float)
min_len = min(len(lb_ml), len(kl_ml))
lb_ml   = lb_ml[:min_len];  kl_ml = kl_ml[:min_len]
mask    = ~(np.isnan(lb_ml) | np.isnan(kl_ml))
lb_ml   = lb_ml[mask];      kl_ml = kl_ml[mask]
try:
    _, p  = wilcoxon(lb_ml, kl_ml)
    sig   = '***' if p < 0.001 else '** ' if p < 0.01 else '*  ' if p < 0.05 else 'n.s.'
    diff  = kl_ml.mean() - lb_ml.mean()
    direction = '↑' if diff > 0 else '↓'
    print(f'  {"KL vs LB CP-gating":<{col}} {"ML Acc":<16} '
          f'{lb_ml.mean():>9.3f}  {kl_ml.mean():>9.3f}  '
          f'{diff:>+8.3f}  {p:>9.4f}  {sig} {direction}')
except ValueError:
    print('  No significant difference in ML accuracy')

print()
print(f'  → KL-reg reduces CP-gating escalation by '
      f'{abs(kl_esc.mean() - lb_esc.mean()):.1f} percentage points vs log-barrier')
print()
print('  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant')
print('  Direction:    ↑ improvement  ↓ worse  '
      '(for escalation rate: ↑ = fewer escalations = better)')

## 📉 Cell 11a — CP Sensitivity Analysis
Sweeps α across both CP-ML and CP-gating to characterise how escalation rate and AUC trade off with the confidence threshold.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Sensitivity Analysis: CP-Gating alpha sweep
# ════════════════════════════════════════════════════════════════════

ALPHA_SENS_GATE = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]
sensitivity_gate = {a: {'esc_rate': [], 'auc_ne': [], 'cp_cov': []} for a in ALPHA_SENS_GATE}

for (seed, fold_idx), data in stored_weights.items():
    w_k      = data['w_k']
    bXcal    = data['bXcal'];    bXval    = data['bXval']
    gcal     = data['gcal'];     gval     = data['gval']
    ycal     = data['ycal'];     yval     = data['yval']
    f_nn_cal = data['f_nn_cal']
    f_nn_val = data['f_nn_val']

    for a in ALPHA_SENS_GATE:
        cds = ConformalDeferralSystem(alpha_gating=a, alpha_ml=a,
                                      cp_gating_on=True, cp_ml_on=False)
        cds.calibrate(w_k, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_k, bXval, gval, f_nn_val)

        esc    = summ['rates'].get('escalate', 0.0)
        cp_cov = compute_conformal_coverage(ml_sets, out, yval, gating_sets=gating_sets)
        non_esc = ~np.isin(out, ['escalate'])
        auc_ne  = (roc_auc_score(yval[non_esc], pred[non_esc])
                   if (non_esc.sum() > 0 and len(np.unique(yval[non_esc])) > 1)
                   else np.nan)

        sensitivity_gate[a]['esc_rate'].append(esc)
        sensitivity_gate[a]['auc_ne'].append(auc_ne)
        sensitivity_gate[a]['cp_cov'].append(cp_cov)

print('\n-- Sensitivity Analysis: KL + CP-Gating only across alpha values --')
print(f"  {'alpha':>6}  {'Esc%':>8}  {'AUC(non-esc)':>12}  {'CPCov':>8}  {'Guarantee':>9}")
print(f"  {'-'*6}  {'-'*8}  {'-'*12}  {'-'*8}  {'-'*9}")
for a in ALPHA_SENS_GATE:
    em = np.nanmean(sensitivity_gate[a]['esc_rate'])
    am = np.nanmean(sensitivity_gate[a]['auc_ne'])
    cm = np.nanmean(sensitivity_gate[a]['cp_cov'])
    print(f"  {a:>6.2f}  {em:>7.1f}%  {am:>12.4f}  {cm:>8.3f}  {1-a:>9.2f}")

print('\n✅ CP-Gating sensitivity analysis complete!')

# ════════════════════════════════════════════════════════════════════
# Sensitivity Analysis: CP-ML alpha sweep
# ════════════════════════════════════════════════════════════════════

sensitivity = {a: {'esc_rate': [], 'auc_ne': [], 'cp_cov': []} for a in ALPHA_SENS}

for (seed, fold_idx), data in stored_weights.items():
    w_k      = data['w_k']
    bXcal    = data['bXcal'];    bXval    = data['bXval']
    gcal     = data['gcal'];     gval     = data['gval']
    ycal     = data['ycal'];     yval     = data['yval']
    f_nn_cal = data['f_nn_cal']
    f_nn_val = data['f_nn_val']

    for a in ALPHA_SENS:
        cds = ConformalDeferralSystem(alpha_ml=a, cp_gating_on=False, cp_ml_on=True)
        cds.calibrate(w_k, bXcal, gcal, ycal, f_nn_cal)
        out, pred, summ, ml_sets, gating_sets = cds.predict(w_k, bXval, gval, f_nn_val)

        esc    = summ['rates'].get('escalate', 0.0)
        cp_cov = compute_conformal_coverage(ml_sets, out, yval, gating_sets=gating_sets)
        non_esc = ~np.isin(out, ['escalate'])   # no ood_abstain in ICU
        auc_ne  = (roc_auc_score(yval[non_esc], pred[non_esc])
                   if (non_esc.sum() > 0 and len(np.unique(yval[non_esc])) > 1)
                   else np.nan)

        sensitivity[a]['esc_rate'].append(esc)
        sensitivity[a]['auc_ne'].append(auc_ne)
        sensitivity[a]['cp_cov'].append(cp_cov)

print('\n── Sensitivity Analysis: KL + CP-ML only across alpha values ──')
print(f"  {'alpha':>6}  {'Esc%':>8}  {'AUC(non-esc)':>12}  {'CPCov':>8}")
print(f"  {'-'*6}  {'-'*8}  {'-'*12}  {'-'*8}")
for a in ALPHA_SENS:
    em = np.nanmean(sensitivity[a]['esc_rate'])
    am = np.nanmean(sensitivity[a]['auc_ne'])
    cm = np.nanmean(sensitivity[a]['cp_cov'])
    print(f'  {a:>6.2f}  {em:>7.1f}%  {am:>12.4f}  {cm:>8.3f}')

print(f'\n  Note: CP coverage should be ≥ 1-α at each row by the conformal guarantee.')
print(f'  Operating alpha (ALPHA_CP={ALPHA_CP}) is highlighted below:')
print(f"  {'→':>2} alpha={ALPHA_CP:.2f}  "
      f"Esc={np.nanmean(sensitivity[ALPHA_CP]['esc_rate']):.1f}%  "
      f"AUC(ne)={np.nanmean(sensitivity[ALPHA_CP]['auc_ne']):.4f}  "
      f"CPCov={np.nanmean(sensitivity[ALPHA_CP]['cp_cov']):.3f}")

print('\n✅ CP-ML sensitivity analysis complete!')

# ── Diagnostic: confirm threshold is near zero (explains 0% escalation) ───────
print('\n── CP-ML threshold diagnostic (first 3 folds) ──')
for (seed, fold_idx), data in list(stored_weights.items())[:3]:
    cds = ConformalDeferralSystem(alpha_ml=ALPHA_CP, cp_gating_on=False, cp_ml_on=True)
    cds.calibrate(data['w_k'], data['bXcal'], data['gcal'], data['ycal'], data['f_nn_cal'])
    print(f'  seed={seed} fold={fold_idx}  CP-ML threshold={cds.cp_ml.threshold:.4f}')
print('  (threshold << 0.5 confirms NN probabilities are concentrated near 0/1)')

## 🔍 Cell 11b — Nonconformity Score Diagnostics
Plots the distribution of calibration nonconformity scores for CP-ML and CP-gating to explain why CP-ML is inactive on MIMIC-III while CP-gating fires meaningfully.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Nonconformity Score Distributions — MIMIC-III ICU 
# ════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('\n── Nonconformity Score Distributions (seed/fold 0) ──')

data      = next(iter(stored_weights.values()))
w_k       = data['w_k']
bXcal     = data['bXcal']
gcal      = data['gcal']
ycal      = data['ycal']
f_nn_cal  = data['f_nn_cal']

cds = ConformalDeferralSystem(alpha_gating=ALPHA_CP, alpha_ml=ALPHA_CP,
                               cp_gating_on=True, cp_ml_on=True)
cds.calibrate(w_k, bXcal, gcal, ycal, f_nn_cal)

BG, PANEL = '#FFFFFF', '#F8F8F8'

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.spines[:].set_color('#CCCCCC')
    ax.tick_params(colors='#333333', labelsize=8)
    ax.xaxis.label.set_color('#222222')
    ax.yaxis.label.set_color('#222222')
    ax.title.set_color('#111111')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(BG)

# ── Panel A: CP-ML nonconformity scores ───────────────────────────────────
style_ax(axes[0])
axes[0].hist(cds.cp_ml.scores, bins=40, color='#5B8DB8', edgecolor='none')
axes[0].axvline(cds.cp_ml.threshold, color='#C0392B', lw=2,
                label=f'Threshold = {cds.cp_ml.threshold:.4f}')
axes[0].set_xlabel('Nonconformity score', fontsize=9)
axes[0].set_ylabel('Count', fontsize=9)
axes[0].set_title(
    '(A)  CP-ML Nonconformity Scores\n'
    '1 − predicted prob of true label',
    fontsize=9.5, fontweight='bold', pad=6,
)
axes[0].legend(fontsize=9)

# ── Panel B: CP-Gating nonconformity scores ───────────────────────────────
style_ax(axes[1])
axes[1].hist(cds.cp_gating.scores, bins=40, color='#2E9E6B', edgecolor='none')
axes[1].axvline(cds.cp_gating.threshold, color='#C0392B', lw=2,
                label=f'Threshold = {cds.cp_gating.threshold:.4f}')
axes[1].set_xlabel('Nonconformity score', fontsize=9)
axes[1].set_ylabel('Count', fontsize=9)
axes[1].set_title(
    '(B)  CP-Gating Nonconformity Scores\n'
    'uncertainty in routing decision',
    fontsize=9.5, fontweight='bold', pad=6,
)
axes[1].legend(fontsize=9)

# ── Panel C: NN predicted probability distribution ────────────────────────
style_ax(axes[2])
axes[2].hist(f_nn_cal, bins=40, color='#5B8DB8', edgecolor='none')
axes[2].axvline(f_nn_cal.mean(), color='#C0392B', lw=2, ls='--',
                label=f'Mean = {f_nn_cal.mean():.4f}')
axes[2].axvline(ycal.mean(), color='#27AE60', lw=2, ls=':',
                label=f'Actual mortality = {ycal.mean():.4f}')
axes[2].set_xlabel('Predicted probability', fontsize=9)
axes[2].set_ylabel('Count', fontsize=9)
axes[2].set_title(
    '(C)  NN Predicted Probability Distribution\n'
    f'spike near 0 — {(f_nn_cal < 0.1).mean()*100:.1f}% of predictions < 0.10',
    fontsize=9.5, fontweight='bold', pad=6,
)
axes[2].legend(fontsize=9)

fig.suptitle(
    'Nonconformity Score Diagnostics — MIMIC-III ICU  (one representative fold)\n'
    f'CP-ML threshold={cds.cp_ml.threshold:.4f}  |  '
    f'CP-Gating threshold={cds.cp_gating.threshold:.4f}  |  '
    f'α={ALPHA_CP}',
    color='#111111', fontsize=11, fontweight='bold', y=1.01,
)

plt.tight_layout()
plt.savefig('mimic_nonconformity_diagnostics.png', dpi=200,
            bbox_inches='tight', facecolor=BG)
plt.show()
plt.close(fig)
print(f'\n✅ Nonconformity diagnostics saved → mimic_nonconformity_diagnostics.png')

# ── Summary statistics ────────────────────────────────────────────────────
print(f'\n  CP-ML     — scores range: [{cds.cp_ml.scores.min():.4f}, '
      f'{cds.cp_ml.scores.max():.4f}]  threshold: {cds.cp_ml.threshold:.4f}')
print(f'  CP-Gating — scores range: [{cds.cp_gating.scores.min():.4f}, '
      f'{cds.cp_gating.scores.max():.4f}]  threshold: {cds.cp_gating.threshold:.4f}')
print(f'\n  NN prediction distribution (calibration set):')
print(f'    Mean predicted prob:  {f_nn_cal.mean():.4f}')
print(f'    Actual mortality:     {ycal.mean():.4f}')
print(f'    Predictions < 0.10:   {(f_nn_cal < 0.1).mean()*100:.1f}%')
print(f'    Predictions > 0.90:   {(f_nn_cal > 0.9).mean()*100:.1f}%')
print(f'\n  If CP-ML threshold << score range minimum → explains 0% escalation')
print(f'  If CP-Gating threshold is within score range → explains active escalation')

## 📊 Cell 11c — NN-Confidence Calibration Analysis
Checks whether the NN's predicted probabilities match observed mortality rates, confirming whether overconfidence is the root cause of CP-ML inactivity.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# NN Calibration Analysis — MIMIC-III ICU  (Cell 31)
# ════════════════════════════════════════════════════════════════════

n_bins    = 10
bin_edges = np.linspace(0, 1, n_bins + 1)

# Collect all validation predictions across all folds
all_preds, all_labels = [], []
for data in stored_weights.values():
    all_preds.append(data['f_nn_val'])
    all_labels.append(data['yval'])
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

# Compute per-bin mean predicted prob and observed frequency
bin_centres, bin_freqs, bin_counts = [], [], []
for i in range(n_bins):
    mask = (all_preds >= bin_edges[i]) & (all_preds < bin_edges[i + 1])
    if mask.sum() > 0:
        bin_centres.append(all_preds[mask].mean())
        bin_freqs.append(all_labels[mask].mean())
        bin_counts.append(mask.sum())

bin_centres = np.array(bin_centres)
bin_freqs   = np.array(bin_freqs)
bin_counts  = np.array(bin_counts)

BG, PANEL = '#FFFFFF', '#F8F8F8'

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.spines[:].set_color('#CCCCCC')
    ax.tick_params(colors='#333333', labelsize=8)
    ax.xaxis.label.set_color('#222222')
    ax.yaxis.label.set_color('#222222')
    ax.title.set_color('#111111')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

# ── Panel A: Reliability diagram ─────────────────────────────────────────
style_ax(axes[0])
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes[0].scatter(
    bin_centres, bin_freqs,
    s=bin_counts / bin_counts.max() * 300,
    color='#5B8DB8', zorder=5, label='NN predictions',
)
axes[0].plot(bin_centres, bin_freqs, color='#5B8DB8', lw=1.5, alpha=0.7)
axes[0].fill_between(
    bin_centres, bin_centres, bin_freqs,
    alpha=0.15, color='#CC4444', label='Overconfidence gap',
)
axes[0].set_xlabel('Mean predicted probability', fontsize=9)
axes[0].set_ylabel('Observed mortality rate', fontsize=9)
axes[0].set_title(
    '(A)  Reliability Diagram — MIMIC-III ICU\n'
    'points below diagonal = overconfident',
    fontsize=9.5, fontweight='bold', pad=6,
)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=8, facecolor='white', labelcolor='#222222',
               edgecolor='#CCCCCC', framealpha=0.92)

# ── Panel B: Prediction distribution ─────────────────────────────────────
style_ax(axes[1])
axes[1].hist(all_preds, bins=40, color='#5B8DB8', edgecolor='none')
axes[1].axvline(all_preds.mean(), color='#C0392B', lw=1.8, ls='--',
                label=f'Mean predicted = {all_preds.mean():.4f}')
axes[1].axvline(all_labels.mean(), color='#27AE60', lw=1.8, ls=':',
                label=f'Actual mortality = {all_labels.mean():.4f}')
axes[1].set_xlabel('Predicted probability', fontsize=9)
axes[1].set_ylabel('Count', fontsize=9)
axes[1].set_title(
    '(B)  NN Prediction Distribution — MIMIC-III ICU\n'
    'spike near 0 explains CP-ML threshold collapse',
    fontsize=9.5, fontweight='bold', pad=6,
)
axes[1].legend(fontsize=8, facecolor='white', labelcolor='#222222',
               edgecolor='#CCCCCC', framealpha=0.92)

fig.suptitle(
    'NN Calibration Analysis — MIMIC-III ICU\n'
    f'ECE = {np.nanmean(records["nn_only"]["ece"]):.4f}  |  '
    f'{(all_preds < 0.1).mean()*100:.1f}% of predictions < 0.10  |  25-fold CV',
    color='#111111', fontsize=11, fontweight='bold', y=1.01,
)

plt.tight_layout()
plt.savefig('mimic_nn_calibration.png', dpi=200, bbox_inches='tight',
            facecolor=BG)
plt.show()
plt.close(fig)
print(f'✅ NN calibration figure saved → mimic_nn_calibration.png')

# Summary stats
print(f'\n  Total predictions:    {len(all_preds):,}')
print(f'  Mean predicted prob:  {all_preds.mean():.4f}')
print(f'  Actual mortality:     {all_labels.mean():.4f}')
print(f'  Predictions > 0.9:    {(all_preds > 0.9).mean()*100:.1f}%')
print(f'  Predictions < 0.1:    {(all_preds < 0.1).mean()*100:.1f}%')
print(f'  ECE (all folds):      {np.nanmean(records["nn_only"]["ece"]):.4f}')

## 📊 Cell 11d — LB vs KL Calibration Comparison
Compares reliability diagrams for NN, LB and KL MoE predictions to investigate why KL-reg produces significant calibration degradation despite superior AUC.

In [ ]:
import matplotlib.pyplot as plt

n_bins    = 10
bin_edges = np.linspace(0, 1, n_bins + 1)

def collect_preds(stored_weights, key):
    """Collect predictions and labels across all folds for a given key."""
    preds, labels = [], []
    for data in stored_weights.values():
        w_k  = data['w_k']
        w_p  = data['w_p']
        bXval = data['bXval']
        gval  = data['gval']
        yval  = data['yval']
        f_nn_val = data['f_nn_val']

        if key == 'nn_only':
            preds.append(f_nn_val)
        elif key == 'pref_lb':
            preds.append(moe_predict_nn(f_nn_val, w_p, gval, bXval))
        elif key == 'pref_kl':
            preds.append(moe_predict_nn(f_nn_val, w_k, gval, bXval))
        labels.append(yval)
    return np.concatenate(preds), np.concatenate(labels)

def reliability_bins(all_preds, all_labels, bin_edges):
    """Compute per-bin mean predicted prob and observed frequency."""
    centres, freqs, counts = [], [], []
    for i in range(len(bin_edges) - 1):
        mask = (all_preds >= bin_edges[i]) & (all_preds < bin_edges[i+1])
        if mask.sum() > 0:
            centres.append(all_preds[mask].mean())
            freqs.append(all_labels[mask].mean())
            counts.append(mask.sum())
    return np.array(centres), np.array(freqs), np.array(counts)

# ── Collect predictions for all three models ─────────────────────
models = {
    'NN Only':          ('nn_only',  '#5B8DB8'),
    'Pref MoE (LB)':   ('pref_lb',  '#2E9E6B'),
    'Pref MoE (KL)':   ('pref_kl',  '#CC4444'),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Reliability Diagrams — NN vs LB vs KL\n'
             'Investigating KL calibration degradation',
             fontsize=13, fontweight='bold')

for col, (label, (key, colour)) in enumerate(models.items()):
    all_preds, all_labels = collect_preds(stored_weights, key)
    centres, freqs, counts = reliability_bins(all_preds, all_labels, bin_edges)

    # Top row: reliability diagram
    ax = axes[0, col]
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
    ax.scatter(centres, freqs, s=counts/counts.max()*300,
               color=colour, zorder=5, label=label)
    ax.plot(centres, freqs, color=colour, lw=1.5, alpha=0.7)
    ax.fill_between(centres, centres, freqs,
                    alpha=0.15, color=colour,
                    label='Calibration gap')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed mortality rate')
    ax.set_title(f'{label}\nReliability Diagram')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

    # Bottom row: prediction histogram
    ax = axes[1, col]
    ax.hist(all_preds, bins=40, color=colour, edgecolor='none', alpha=0.8)
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Count')
    ax.set_title(f'{label}\nPrediction Distribution')

    # Print summary stats
    ece = np.nanmean(records[key]['ece'])
    brier = np.nanmean(records[key]['brier'])
    print(f'\n  {label}')
    print(f'    ECE:               {ece:.4f}')
    print(f'    Brier:             {brier:.4f}')
    print(f'    Mean pred prob:    {all_preds.mean():.4f}')
    print(f'    Actual mortality:  {all_labels.mean():.4f}')
    print(f'    Preds > 0.9:       {(all_preds > 0.9).mean()*100:.1f}%')
    print(f'    Preds < 0.1:       {(all_preds < 0.1).mean()*100:.1f}%')

plt.tight_layout()
plt.show()